# Run `amir/score.py` in Colab

Open this notebook in Google Colab, then choose `Runtime -> Change runtime type` and select a GPU runtime.

This workflow is wired for the `amir/transcripts` corpus. GPU is supported through PyTorch + Transformers. TPU runtimes can still open the notebook, but `score.py` currently falls back to CPU unless you add `torch-xla` support.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Ngafney/garda-spring26.git"
REPO_DIR = Path("/content/garda-spring26") if IN_COLAB else Path.cwd()
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_REPO_DIR = Path("/content/drive/MyDrive/garda-spring26")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = GOOGLE_DRIVE_REPO_DIR

if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {REPO_DIR}")

Working directory: /content/garda-spring26


In [2]:
%pip install -q pandas tqdm torch transformers tomli

In [3]:
import os
import sys
import torch

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
if os.environ.get("COLAB_TPU_ADDR"):
    print("TPU runtime detected. score.py will still use CPU unless torch-xla support is added.")

Python: 3.12.13
CUDA available: True
CUDA device count: 1
GPU: Tesla T4


## Parameters

Set `LIMIT = None` for the full corpus. There are about 9,951 transcript files in `amir/transcripts`, so a full run can take a long time even on Colab GPU.

Use `PATTERN` to score a subset first, for example `"amazon"` or `"nvidia"`.

If you already have `amir/transcripts_metadata.csv`, leave `REBUILD_METADATA = False` and the notebook will use that file.

Set `SAVE_OUTPUTS_TO_DRIVE = True` if you want the CSV and SQLite cache to survive a Colab runtime reset.

In [4]:
TRANSCRIPTS_DIR = REPO_DIR / "amir" / "transcripts"
METADATA_CSV = REPO_DIR / "amir" / "transcripts_metadata.csv"

SAVE_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/garda_outputs")
LOCAL_OUTPUT_DIR = REPO_DIR / "amir" / "outputs"

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR
else:
    OUTPUT_DIR = LOCAL_OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "transcript_scores.csv"
SCORE_DB = OUTPUT_DIR / "score_cache.sqlite"

LIMIT = None
PATTERN = None
FORCE = True
REBUILD_METADATA = False
VERBOSE = 1

print(TRANSCRIPTS_DIR)
print(METADATA_CSV)
print(OUTPUT_CSV)


/content/garda-spring26/amir/transcripts
/content/garda-spring26/amir/transcripts_metadata.csv
/content/drive/MyDrive/garda_outputs/transcript_scores.csv


In [5]:
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "amir.score",
    "--transcripts-dir",
    str(TRANSCRIPTS_DIR),
    "--metadata-csv",
    str(METADATA_CSV),
    "--output-csv",
    str(OUTPUT_CSV),
    "--score-db",
    str(SCORE_DB),
]

if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]
if PATTERN:
    cmd += ["--pattern", PATTERN]
if REBUILD_METADATA:
    cmd.append("--rebuild-metadata")
if FORCE:
    cmd.append("--force")
cmd += ["-v"] * VERBOSE

print("Running:")
print(" ".join(shlex.quote(part) for part in cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"score.py failed with exit code {result.returncode}")


Running:
/usr/bin/python3 -m amir.score --transcripts-dir /content/garda-spring26/amir/transcripts --metadata-csv /content/garda-spring26/amir/transcripts_metadata.csv --output-csv /content/garda-spring26/amir/outputs/transcript_scores.csv --score-db /content/garda-spring26/amir/outputs/score_cache.sqlite --force -v
Wrote 9,954 scored transcripts to /content/garda-spring26/amir/outputs/transcript_scores.csv



In [6]:
import pandas as pd

scores = pd.read_csv(OUTPUT_CSV)
print(f"Rows written: {len(scores):,}")
display(scores.head())

Rows written: 9,954


,transcript_id,company,ticker,country,call_date,quarter,fiscal_year,source_url,transcript_path,word_count,...,ai_labor_score,lm_positive_density,lm_negative_density,lm_uncertainty_density,lm_net_tone,guidance_raised,guidance_lowered,management_confidence_score,risk_mentions_count,scored_at
0,a9d1176a-10c0-56d3-bfbd-77c6ac37c071,Blue Bird,BLBD,Unknown,2026-04-10,Q1,2025,https://www.fool.com/earnings/call-transcripts...,/content/garda-spring26/amir/transcripts/blue-...,168,...,0.000000,0.005882,0.000000,0.000000,0.005882,False,False,1.000000,0,2026-04-14T04:38:47+00:00
1,30b96fe9-0bd1-513b-8d9d-aea8dc085bca,Blue Bird,BLBD,Unknown,2026-04-10,Q3,2025,https://www.fool.com/earnings/call-transcripts...,/content/garda-spring26/amir/transcripts/blue-...,812,...,0.064824,0.006075,0.004860,0.002430,0.001215,False,False,0.076923,6,2026-04-14T04:38:47+00:00
2,f71398c8-13e1-509c-bdac-1726be1cb3a8,China Auto,CAAS,Unknown,2026-04-10,Q4,2024,https://www.fool.com/earnings/call-transcripts...,/content/garda-spring26/amir/transcripts/china...,2799,...,0.000000,0.004095,0.001489,0.001862,0.002606,False,False,0.333333,1,2026-04-14T04:38:47+00:00
3,f1ee0cc8-de31-5277-8034-8b8cc12262b4,China Auto Systems Earnings Transcript,CAAS,Unknown,2026-04-10,UNKNOWN,2026,https://www.fool.com/earnings/call-transcripts...,/content/garda-spring26/amir/transcripts/china...,3621,...,0.000000,0.002849,0.002564,0.002849,0.000285,True,False,-0.500000,6,2026-04-14T04:38:48+00:00
4,f1ee0cc8-de31-5277-8034-8b8cc12262b4,China Automotive,CAAS,Unknown,2026-04-10,UNKNOWN,2026,https://www.fool.com/earnings/call-transcripts...,/content/garda-spring26/amir/transcripts/china...,3762,...,0.232521,0.001906,0.000817,0.001634,0.001089,True,False,0.333333,1,2026-04-14T04:38:48+00:00


In [ ]:
from google.colab import files

files.download(str(OUTPUT_CSV))


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

drive_dir = Path("/content/drive/MyDrive/garda_outputs")
drive_dir.mkdir(parents=True, exist_ok=True)

target = drive_dir / scores.name
shutil.copy2(scores, target)

print(target)


Mounted at /content/drive


NameError: name 'OUTPUT_CSV' is not defined